# aeon-qc exploration

Interactive cells for testing QC metrics against benchmark datasets.
See `BENCHMARKS.md` for dataset paths and time ranges.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from aeon_qc import heartbeat_gaps, dropped_frames, epoch_gaps, run_qc, save_results, generate_report, schema_from_metadata

## Configuration

Edit these values to point at the dataset you want to inspect.

In [ ]:
ROOT = r"Z:\aeon\data\raw\AEON4\social0.4"
SCHEMA = schema_from_metadata(ROOT)  # reads workflow tag from Metadata.yml â†’ full social04 schema

START = pd.Timestamp("2024-09-13T12:00:00", tz="UTC")
END   = pd.Timestamp("2024-09-13T13:00:00", tz="UTC")

print("Schema devices:")
for name, streams in SCHEMA.items():
    if isinstance(streams, dict):
        print(f"  {name}: {list(streams.keys())}")
    else:
        print(f"  {name}: {type(streams).__name__}")

In [ ]:
from swc.aeon.io.api import load
from swc.aeon.io.reader import Metadata as MetadataReader

# Devices listed in Metadata.yml for this dataset
meta_df = load(ROOT, MetadataReader())
meta_devices = set(meta_df.iloc[0].metadata.Devices.keys())

# Top-level device names in SCHEMA (excluding the Metadata reader entry)
schema_devices = {name for name in SCHEMA.keys() if name != "Metadata"}

only_in_meta = meta_devices - schema_devices
only_in_schema = schema_devices - meta_devices

print(f"Metadata.yml devices : {len(meta_devices)}")
print(f"Schema devices       : {len(schema_devices)}")
print(f"Matched              : {len(meta_devices & schema_devices)}")

if only_in_meta:
    print(f"\nIn Metadata only (not covered by schema): {sorted(only_in_meta)}")
if only_in_schema:
    print(f"\nIn schema only (not in Metadata.yml):     {sorted(only_in_schema)}")
if not only_in_meta and not only_in_schema:
    print("\nAll devices match.")

## Epoch gaps

Detects gaps between consecutive Bonsai session starts within the `START`/`END` window.
`load()` assembles `Metadata.yml` timestamps across all epoch directories in that range,
so a short gap suggests a crash/restart and a long gap a planned stop â€” interpret in context
of your experiment schedule.

In [ ]:
epochs = epoch_gaps(ROOT, start=START, end=END)
print(f"{len(epochs) + 1} epoch(s) in range, {len(epochs)} inter-epoch gap(s)")
if not epochs.empty:
    print(f"Shortest: {epochs['gap_duration'].min()}, Longest: {epochs['gap_duration'].max()}")
    display(epochs)

## Heartbeat gaps

Detects periods where a Harp device stops sending heartbeat events.
Returns a DataFrame with `gap_end`, `duration`, `device` columns.

In [ ]:
from swc.aeon.io.reader import Heartbeat as HeartbeatReader
from aeon_qc.report import iter_readers

hb_devices = {name: reader for name, reader in iter_readers(SCHEMA)
              if isinstance(reader, HeartbeatReader)}

if not hb_devices:
    print("No Heartbeat devices found in this dataset")
else:
    print(f"Found {len(hb_devices)} heartbeat device(s): {list(hb_devices.keys())}\n")
    for device_name, hb_reader in hb_devices.items():
        gaps = heartbeat_gaps(ROOT, hb_reader, start=START, end=END)
        data_ok = gaps.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA on disk")
        elif gaps.empty:
            print(f"{device_name}: 0 gap(s)")
        else:
            print(f"{device_name}: {len(gaps)} gap(s)")
            display(gaps)

In [ ]:
from aeon_qc import sync_delta
from aeon_qc.sync import MIN_DEVICES

if len(hb_devices) < MIN_DEVICES:
    print(f"sync_delta: need at least {MIN_DEVICES} heartbeat devices (found {len(hb_devices)})")
else:
    delta = sync_delta(ROOT, hb_devices, start=START, end=END)
    if delta.empty:
        print("sync_delta: no data")
    else:
        summary = (
            delta.groupby("device")["delta_seconds"]
            .agg(max_abs=lambda s: s.abs().max(), mean_abs=lambda s: s.abs().mean())
            .sort_values("max_abs", ascending=False)
        )
        print("Sync delta per device (seconds):")
        print(summary.to_string())

## Dropped frames

Detects dropped video frames via `hw_counter` jumps.
Returns a DataFrame with `n_dropped`, `hw_counter_before`, `hw_counter_after`, `device`.

In [ ]:
from swc.aeon.io.reader import Video as VideoReader

camera_devices = {name: reader for name, reader in iter_readers(SCHEMA)
                  if isinstance(reader, VideoReader)}

if not camera_devices:
    print("No camera devices found in this dataset")
else:
    print(f"Found {len(camera_devices)} camera(s): {list(camera_devices.keys())}\n")
    for device_name, cam_reader in camera_devices.items():
        drops = dropped_frames(ROOT, cam_reader, start=START, end=END)
        data_ok = drops.attrs.get("data_found", True)
        if not data_ok:
            print(f"{device_name}: NO DATA")
        elif drops.empty:
            print(f"{device_name}: 0 drop event(s)")
        else:
            total = drops["n_dropped"].sum()
            print(f"{device_name}: {len(drops)} drop event(s), {total} total frames dropped")
            display(drops)